## Load Dataset

In [ ]:
!git clone https://github.com/tajuar-akash-hub/Datasets 

Cloning into 'Datasets'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 66 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 5.63 MiB | 6.11 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [ ]:
df['question'][0]

'What is the capital of "France"?'

## Tokenize

In [11]:
import re
def tokenize(text):
  #1 Lowercase
  text = text.lower()

  #2 remove quotes
  text = re.sub(r" [\"']", "", text)

  #3 remove all punctuations
  text = re.sub(r"[^a-z0-9\s]", " ", text)

  #4 remove extra space
  text = re.sub(r"\s+", " ", text).strip()

  # tokenize split into words
  tokens = text.split()

  return tokens

In [12]:
print(tokenize(df['question'][0]))

['what', 'is', 'the', 'capital', 'offrance']


## Vocab forming

In [17]:
vocab = {'<UNK>':0}

In [26]:
def build_vocab(row):
  # print(row['question'], row['answer'])

  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  # print(tokenized_question, tokenized_answer)

  merged_tokens = tokenized_question + tokenized_answer

  # print(merged_tokens)

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)


In [27]:
df.apply(build_vocab, axis = 1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [28]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'offrance': 5,
 'paris': 6,
 'ofgermany': 7,
 'berlin': 8,
 'ofitaly': 9,
 'rome': 10,
 'ofspain': 11,
 'madrid': 12,
 'ofjapan': 13,
 'tokyo': 14,
 'ofcanada': 15,
 'ottawa': 16,
 'ofbrazil': 17,
 'brasilia': 18,
 'ofaustralia': 19,
 'canberra': 20,
 'ofindia': 21,
 'new': 22,
 'delhi': 23,
 'ofchina': 24,
 'beijing': 25,
 'ofrussia': 26,
 'moscow': 27,
 'ofunited': 28,
 'states': 29,
 'washington': 30,
 'dc': 31,
 'ofmexico': 32,
 'mexico': 33,
 'city': 34,
 'ofegypt': 35,
 'cairo': 36,
 'ofturkey': 37,
 'ankara': 38,
 'ofargentina': 39,
 'buenos': 40,
 'aires': 41,
 'ofsouth': 42,
 'korea': 43,
 'seoul': 44,
 'ofindonesia': 45,
 'jakarta': 46,
 'ofpakistan': 47,
 'islamabad': 48,
 'ofbangladesh': 49,
 'dhaka': 50,
 'ofnepal': 51,
 'kathmandu': 52,
 'ofsri': 53,
 'lanka': 54,
 'colombo': 55,
 'ofthailand': 56,
 'bangkok': 57,
 'ofmalaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'ofvietnam': 61,
 'hanoi': 62,
 'ofuae': 63,
 'ab

In [29]:
len(vocab)

246

In [30]:
vocab['what']

1

### text to indices

In [36]:
def text_to_indices(text, vocab):
  indexed_text = []

  for token in tokenize(text):
    # print(token)
    if token in vocab:
      # print(vocab[token])
      indexed_text.append(vocab[token])

    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [37]:
text_to_indices("the is Phitron", vocab)

[3, 2, 0]

## Dataset and Dataloader

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

In [39]:
df.shape[0]

171

In [45]:
index = 0
text_to_indices(df.iloc[index]['question'],vocab)

[1, 2, 3, 4, 5]

In [46]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [47]:
dataset = QADataset(df, vocab)

In [48]:
dataset[1]

(tensor([1, 2, 3, 4, 7]), tensor([8]))

In [49]:
dataloader = DataLoader(dataset, batch_size=1, shuffle = True)

In [50]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[ 94, 235, 236,  92,  93,  94,  95,  96]]) tensor([97, 98])
tensor([[237, 238, 239,  79,  83,  84]]) tensor([85, 86, 87])
tensor([[233, 234,  79,  83, 159, 160]]) tensor([161, 162])
tensor([[233, 234,   1, 138, 132, 121]]) tensor([139, 130, 134])
tensor([[237, 238, 239,   1,  88,  89,  90]]) tensor([91])
tensor([[79, 83, 84]]) tensor([85, 86, 87])
tensor([[237, 238, 239,  79,  83, 163]]) tensor([164, 165])
tensor([[ 1, 88, 89, 90]]) tensor([91])
tensor([[233, 234,  79, 171, 172]]) tensor([173, 174, 175])
tensor([[ 1,  2,  3,  4, 13]]) tensor([14])
tensor([[240,   1, 241, 242,  92, 181,   2,   3, 110, 185, 186]]) tensor([187, 188])
tensor([[ 92, 195,   2, 196, 197]]) tensor([198])
tensor([[ 79, 113, 114, 115, 116]]) tensor([117, 118])
tensor([[1, 2, 3, 4, 9]]) tensor([10])
tensor([[ 1,  2,  3,  4, 73]]) tensor([74])
tensor([[233, 234,  92, 181,   2, 100, 121, 193]]) tensor([194])
tensor([[ 1,  2,  3,  4, 63]]) tensor([64, 65])
tensor([[79, 80]]) tensor([81, 82])
tensor([[  1,   

In [51]:
# Squeeze
import torch
x = torch.tensor([[[1,2,3]]])

# print(x.shape)

y = x.squeeze(0)
print(y.shape)

torch.Size([1, 1, 3])


### RNN Architecture Implementation

In [52]:
import torch.nn as nn
class simpleRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim =50, hidden_size=64):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, question):
    embedded = self.embedding(question) # (1, seq_len, 50)
    _,final = self.rnn(embedded)
    return self.fc(final.squeeze(0)) # (1, vocab_size)


In [53]:
model = simpleRNN(vocab_size = len(vocab))

In [54]:
# Example of unsqueeze
import torch
x = torch.tensor([1,2,3]) # shape: (3,)
print(x.shape)

y = x.unsqueeze(0)
print(y.shape)

torch.Size([3])
torch.Size([1, 3])


### Traning loop

In [55]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

In [56]:
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()
    output = model(question)        # (1, vocab_size)

    # Fix: take only the first answer token as target -> shape(1,)
    target = answer[0][0].unsqueeze(0)

    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch+1:3d}/{epochs} | Loss {total_loss:.4f}")

Epoch  10/50 
Epoch  20/50 
Epoch  30/50 
Epoch  40/50 
Epoch  50/50 


### Prediction

In [57]:
vocab.items()

dict_items([('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('offrance', 5), ('paris', 6), ('ofgermany', 7), ('berlin', 8), ('ofitaly', 9), ('rome', 10), ('ofspain', 11), ('madrid', 12), ('ofjapan', 13), ('tokyo', 14), ('ofcanada', 15), ('ottawa', 16), ('ofbrazil', 17), ('brasilia', 18), ('ofaustralia', 19), ('canberra', 20), ('ofindia', 21), ('new', 22), ('delhi', 23), ('ofchina', 24), ('beijing', 25), ('ofrussia', 26), ('moscow', 27), ('ofunited', 28), ('states', 29), ('washington', 30), ('dc', 31), ('ofmexico', 32), ('mexico', 33), ('city', 34), ('ofegypt', 35), ('cairo', 36), ('ofturkey', 37), ('ankara', 38), ('ofargentina', 39), ('buenos', 40), ('aires', 41), ('ofsouth', 42), ('korea', 43), ('seoul', 44), ('ofindonesia', 45), ('jakarta', 46), ('ofpakistan', 47), ('islamabad', 48), ('ofbangladesh', 49), ('dhaka', 50), ('ofnepal', 51), ('kathmandu', 52), ('ofsri', 53), ('lanka', 54), ('colombo', 55), ('ofthailand', 56), ('bangkok', 57), ('ofmalaysia', 58), ('kuala'

In [58]:
idx_to_word = {idx : word for word, idx in vocab.items()}
idx_to_word

{0: '<UNK>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'offrance',
 6: 'paris',
 7: 'ofgermany',
 8: 'berlin',
 9: 'ofitaly',
 10: 'rome',
 11: 'ofspain',
 12: 'madrid',
 13: 'ofjapan',
 14: 'tokyo',
 15: 'ofcanada',
 16: 'ottawa',
 17: 'ofbrazil',
 18: 'brasilia',
 19: 'ofaustralia',
 20: 'canberra',
 21: 'ofindia',
 22: 'new',
 23: 'delhi',
 24: 'ofchina',
 25: 'beijing',
 26: 'ofrussia',
 27: 'moscow',
 28: 'ofunited',
 29: 'states',
 30: 'washington',
 31: 'dc',
 32: 'ofmexico',
 33: 'mexico',
 34: 'city',
 35: 'ofegypt',
 36: 'cairo',
 37: 'ofturkey',
 38: 'ankara',
 39: 'ofargentina',
 40: 'buenos',
 41: 'aires',
 42: 'ofsouth',
 43: 'korea',
 44: 'seoul',
 45: 'ofindonesia',
 46: 'jakarta',
 47: 'ofpakistan',
 48: 'islamabad',
 49: 'ofbangladesh',
 50: 'dhaka',
 51: 'ofnepal',
 52: 'kathmandu',
 53: 'ofsri',
 54: 'lanka',
 55: 'colombo',
 56: 'ofthailand',
 57: 'bangkok',
 58: 'ofmalaysia',
 59: 'kuala',
 60: 'lumpur',
 61: 'ofvietnam',
 62: 'hanoi',
 63: 'ofuae',
 64:

In [63]:
def predict(question_text, model, vocab, idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text, vocab)

    if not indices:
      return "<UNK>"

    x = torch.tensor(indices).unsqueeze(0)  #batch size, sequence length
    pred = torch.argmax(model(x), dim=1).item()

  return idx_to_word.get(pred, "<UNK>")

In [65]:
# Test
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",
]
for q in tests:
  print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")



What is the capital of France?                → japan
What is H2O commonly called?                  → water
Which planet is known as the red planet?      → mars
Which bird cannot fly?                        → ostrich
What is the capital of Japan ?                → pacific
